# CNN Replica Notebook — split first, then augment

This version fixes augmented-sibling leakage.

Pipeline:

1. Load only the original CSV rows from `Kefra_Processed_Data`.
2. Create a stratified 90/10 split on the original rows.
3. Augment only the training split by default.
4. Test on untouched original holdout rows by default.
5. Fit the scaler on the augmented training split only.

This means no augmented version of a test signal can appear in training.

Optional: set `AUGMENT_TEST_SET = True` if you want a larger test confusion matrix made from augmented copies of the held-out originals. That still avoids train/test sibling leakage, because the held-out originals are split before augmentation.


In [1]:
# ==============================
# Block 1: Load originals, split 90/10, then augment training only
# ==============================

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

# ------------------------------
# Reproducibility
# ------------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ------------------------------
# Paths and split/augmentation options
# ------------------------------

RAW_DATA_DIR = Path("Kefra_Processed_Data")

# Important options:
# - INCLUDE_ORIGINAL_TRAIN=False gives the paper-style 6x expansion:
#       306 original train rows -> 1836 augmented train rows, if the full 340-row dataset is present.
# - INCLUDE_ORIGINAL_TRAIN=True gives original + six augmentations, a 7x train set.
# - AUGMENT_TEST_SET=False is the cleanest evaluation: test on untouched original holdout rows.
# - AUGMENT_TEST_SET=True makes a larger no-leakage test set by augmenting only the held-out originals.
INCLUDE_ORIGINAL_TRAIN = True
AUGMENT_TEST_SET = False

# 1D augmentation settings, matching augment_kefra_csv.py defaults.
Y_TOP = -10.0
Y_BOTTOM = -130.0
ROTATION_DEGREES = 10.0
SCALE_RANGE = (0.90, 1.10)
HSHIFT_FRAC = 0.10
VSHIFT_VALUE = 5.0
N_FEATURES = 512

# ------------------------------
# 1D augmentation helpers
# ------------------------------

def resample_1d(y: np.ndarray, n: int) -> np.ndarray:
    old_x = np.linspace(0.0, 1.0, len(y))
    new_x = np.linspace(0.0, 1.0, n)
    return np.interp(new_x, old_x, y)


def horizontal_shift(y: np.ndarray, frac_shift: float) -> np.ndarray:
    n = len(y)
    x = np.arange(n, dtype=float)
    src_x = x - frac_shift * n
    return np.interp(src_x, x, y, left=y[0], right=y[-1])


def vertical_reflect(y: np.ndarray, y_top: float | None = Y_TOP, y_bottom: float | None = Y_BOTTOM) -> np.ndarray:
    if y_top is not None and y_bottom is not None:
        center_twice = y_top + y_bottom
    else:
        center_twice = float(np.nanmin(y) + np.nanmax(y))
    return center_twice - y


def rotation_like_warp(y: np.ndarray, degrees: float) -> np.ndarray:
    """Approximate image rotation for a 1D plotted waveform."""
    n = len(y)
    x = np.linspace(-1.0, 1.0, n)
    ymin, ymax = float(np.nanmin(y)), float(np.nanmax(y))
    scale = ymax - ymin
    if scale < 1e-12:
        return y.copy()

    y_norm = 2.0 * (y - ymin) / scale - 1.0
    theta = np.deg2rad(degrees)

    xr = x * np.cos(theta) - y_norm * np.sin(theta)
    yr = x * np.sin(theta) + y_norm * np.cos(theta)

    order = np.argsort(xr)
    xr = xr[order]
    yr = yr[order]

    target_x = np.linspace(max(-1.0, xr.min()), min(1.0, xr.max()), n)
    y_interp = np.interp(target_x, xr, yr, left=yr[0], right=yr[-1])
    y_out = (y_interp + 1.0) * 0.5 * scale + ymin
    return resample_1d(y_out, n)


def rescale_waveform(y: np.ndarray, x_scale: float, y_scale: float) -> np.ndarray:
    n = len(y)
    x = np.linspace(-1.0, 1.0, n)
    src_x = x / x_scale
    old_x = np.linspace(-1.0, 1.0, n)
    y_x_scaled = np.interp(src_x, old_x, y, left=y[0], right=y[-1])
    center = float(np.nanmean(y_x_scaled))
    return center + y_scale * (y_x_scaled - center)


def augment_one_row(y: np.ndarray, rng: np.random.Generator) -> list[tuple[str, np.ndarray]]:
    """Return exactly six paper-style augmentations for one amplitude vector."""
    rot = rng.uniform(-ROTATION_DEGREES, ROTATION_DEGREES)
    x_scale = rng.uniform(SCALE_RANGE[0], SCALE_RANGE[1])
    y_scale = rng.uniform(SCALE_RANGE[0], SCALE_RANGE[1])
    hshift = rng.uniform(-HSHIFT_FRAC, HSHIFT_FRAC)
    vshift = rng.uniform(-VSHIFT_VALUE, VSHIFT_VALUE)

    return [
        ("x_reflect", y[::-1].copy()),
        ("y_reflect", vertical_reflect(y)),
        ("rotation", rotation_like_warp(y, rot)),
        ("rescale", rescale_waveform(y, x_scale, y_scale)),
        ("h_translate", horizontal_shift(y, hshift)),
        ("v_translate", y + vshift),
    ]


def augment_split(X_in, y_in, files_in, seed: int, include_original: bool, split_name: str):
    """Augment a split after the split has already been made."""
    rng = np.random.default_rng(seed)
    out_X = []
    out_y = []
    out_files = []
    out_base_files = []

    files_in = pd.Series(files_in).reset_index(drop=True)

    for i, row in enumerate(X_in):
        base_file = str(files_in.iloc[i])
        label = int(y_in[i])

        if include_original:
            out_X.append(row.astype(np.float32).copy())
            out_y.append(label)
            out_files.append(f"{base_file}__{split_name}__original")
            out_base_files.append(base_file)

        for aug_name, aug_row in augment_one_row(row.astype(np.float32), rng):
            out_X.append(aug_row.astype(np.float32))
            out_y.append(label)
            out_files.append(f"{base_file}__{split_name}__aug_{aug_name}")
            out_base_files.append(base_file)

    return (
        np.vstack(out_X).astype(np.float32),
        np.array(out_y, dtype=np.int64),
        pd.Series(out_files),
        pd.Series(out_base_files),
    )

# ------------------------------
# Load original CSV files only
# ------------------------------

df_r = pd.read_csv(RAW_DATA_DIR / "Real.csv")
df_fh = pd.read_csv(RAW_DATA_DIR / "Fake_High.csv")
df_fl = pd.read_csv(RAW_DATA_DIR / "Fake_Low.csv")

print("Original Real:", df_r.shape)
print("Original Fake High:", df_fh.shape)
print("Original Fake Low:", df_fl.shape)

files_r = df_r["file"].copy() if "file" in df_r.columns else pd.Series([f"real_{i}" for i in range(len(df_r))])
files_fh = df_fh["file"].copy() if "file" in df_fh.columns else pd.Series([f"fake_high_{i}" for i in range(len(df_fh))])
files_fl = df_fl["file"].copy() if "file" in df_fl.columns else pd.Series([f"fake_low_{i}" for i in range(len(df_fl))])

X_real = df_r.drop(columns=["file"], errors="ignore").to_numpy(dtype=np.float32)
X_fh = df_fh.drop(columns=["file"], errors="ignore").to_numpy(dtype=np.float32)
X_fl = df_fl.drop(columns=["file"], errors="ignore").to_numpy(dtype=np.float32)

assert X_real.shape[1] == N_FEATURES, f"Expected {N_FEATURES} columns, got {X_real.shape[1]}"
assert X_fh.shape[1] == N_FEATURES, f"Expected {N_FEATURES} columns, got {X_fh.shape[1]}"
assert X_fl.shape[1] == N_FEATURES, f"Expected {N_FEATURES} columns, got {X_fl.shape[1]}"

# Labels: 0=Real, 1=Fake High, 2=Fake Low
X_original = np.vstack([X_real, X_fh, X_fl]).astype(np.float32)
y_original = np.concatenate([
    np.zeros(len(X_real), dtype=np.int64),
    np.ones(len(X_fh), dtype=np.int64),
    np.full(len(X_fl), 2, dtype=np.int64)
])
files_original = pd.concat([files_r, files_fh, files_fl], ignore_index=True)

class_names = ["Real", "Fake High", "Fake Low"]

print("Original combined X:", X_original.shape)
print("Original class counts:", pd.Series(y_original).value_counts().sort_index().to_dict())

# ------------------------------
# Split original rows first: this is the leakage-prevention step
# ------------------------------

X_train_orig, X_test_orig, y_train_orig, y_test_orig, files_train_orig, files_test_orig = train_test_split(
    X_original,
    y_original,
    files_original,
    test_size=0.10,
    random_state=SEED,
    shuffle=True,
    stratify=y_original,
)

print("\nAfter original 90/10 split:")
print("Train originals:", X_train_orig.shape, pd.Series(y_train_orig).value_counts().sort_index().to_dict())
print("Test originals:", X_test_orig.shape, pd.Series(y_test_orig).value_counts().sort_index().to_dict())

# ------------------------------
# Augment after the split
# ------------------------------

X_train, y_train, files_train, train_base_files = augment_split(
    X_train_orig,
    y_train_orig,
    files_train_orig,
    seed=SEED + 100,
    include_original=INCLUDE_ORIGINAL_TRAIN,
    split_name="train",
)

if AUGMENT_TEST_SET:
    X_test, y_test, files_test, test_base_files = augment_split(
        X_test_orig,
        y_test_orig,
        files_test_orig,
        seed=SEED + 200,
        include_original=False,
        split_name="test",
    )
else:
    X_test = X_test_orig.astype(np.float32)
    y_test = y_test_orig.astype(np.int64)
    files_test = pd.Series(files_test_orig).reset_index(drop=True)
    test_base_files = pd.Series(files_test_orig).reset_index(drop=True)

# Keep aliases for compatibility with later code and for CV.
X = X_original
y = y_original
files = files_original.reset_index(drop=True)

print("\nFinal train/test used by Block 3:")
print("X_train:", X_train.shape, pd.Series(y_train).value_counts().sort_index().to_dict())
print("X_test:", X_test.shape, pd.Series(y_test).value_counts().sort_index().to_dict())
print("INCLUDE_ORIGINAL_TRAIN:", INCLUDE_ORIGINAL_TRAIN)
print("AUGMENT_TEST_SET:", AUGMENT_TEST_SET)

# Leakage check: no original source file should be present in both train and test.
train_sources = set(train_base_files.astype(str))
test_sources = set(test_base_files.astype(str))
overlap = train_sources.intersection(test_sources)
print("\nLeakage check: shared original source files between train and test =", len(overlap))
if overlap:
    raise RuntimeError(f"Leakage detected: {list(sorted(overlap))[:10]}")
else:
    print("No augmented-sibling leakage: split happened before augmentation.")


Original Real: (110, 513)
Original Fake High: (110, 513)
Original Fake Low: (120, 513)
Original combined X: (340, 512)
Original class counts: {0: 110, 1: 110, 2: 120}

After original 90/10 split:
Train originals: (306, 512) {0: 99, 1: 99, 2: 108}
Test originals: (34, 512) {0: 11, 1: 11, 2: 12}

Final train/test used by Block 3:
X_train: (2142, 512) {0: 693, 1: 693, 2: 756}
X_test: (34, 512) {0: 11, 1: 11, 2: 12}
INCLUDE_ORIGINAL_TRAIN: True
AUGMENT_TEST_SET: False

Leakage check: shared original source files between train and test = 0
No augmented-sibling leakage: split happened before augmentation.


In [2]:
# ==============================
# Block 2: 1D CNN model, training, testing functions
# ==============================

import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

BATCH_SIZE = 8          # same as paper
LEARNING_RATE = 1e-3    # same as paper
NUM_EPOCHS = 100        # same as paper


def make_activation(name: str):
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "leakyrelu":
        return nn.LeakyReLU(negative_slope=0.01)
    if name == "elu":
        return nn.ELU()
    raise ValueError(f"Unknown activation: {name}")


class CNN1DClassifier(nn.Module):
    """Small 1D CNN classifier for 512-point amplitude vectors."""

    def __init__(self, num_classes=3, activation="relu"):
        super().__init__()
        act1 = make_activation(activation)
        act2 = make_activation(activation)
        act3 = make_activation(activation)
        act4 = make_activation(activation)

        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),   # 512 -> 256
            act1,
            nn.BatchNorm1d(16),

            nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2),  # 256 -> 128
            act2,
            nn.BatchNorm1d(32),

            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),  # 128 -> 64
            act3,
            nn.BatchNorm1d(64),

            nn.Conv1d(64, 128, kernel_size=3, stride=2, padding=1), # 64 -> 32
            act4,
            nn.BatchNorm1d(128),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            make_activation(activation),
            nn.Dropout(0.20),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def make_optimizer(name: str, model: nn.Module, lr: float):
    name = name.lower()
    if name == "sgd":
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    if name == "adam":
        return optim.Adam(model.parameters(), lr=lr)
    if name == "rmsprop":
        return optim.RMSprop(model.parameters(), lr=lr, momentum=0.9)
    raise ValueError(f"Unknown optimizer: {name}")


def scale_and_tensorize(X_train, X_test):
    # Fit scaler on training only to avoid leakage.
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train).astype(np.float32)
    X_test_s = scaler.transform(X_test).astype(np.float32)

    # Conv1d expects [batch, channels, length].
    X_train_t = torch.tensor(X_train_s).unsqueeze(1)
    X_test_t = torch.tensor(X_test_s).unsqueeze(1)
    return X_train_t, X_test_t, scaler


def make_loader(X_tensor, y_array, batch_size=BATCH_SIZE, shuffle=False):
    y_tensor = torch.tensor(y_array, dtype=torch.long)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)


def train_model(X_train, y_train, X_test, y_test, optimizer_name="sgd", activation="relu", verbose=False):
    X_train_t, X_test_t, scaler = scale_and_tensorize(X_train, X_test)
    train_loader = make_loader(X_train_t, y_train, shuffle=True)
    test_loader = make_loader(X_test_t, y_test, shuffle=False)

    model = CNN1DClassifier(num_classes=3, activation=activation).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(optimizer_name, model, LEARNING_RATE)

    train_losses = []

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f"Epoch {epoch+1:03d}/{NUM_EPOCHS} | loss={epoch_loss:.6f}")

    y_pred, y_prob = predict_model(model, test_loader)

    return {
        "model": model,
        "scaler": scaler,
        "train_losses": train_losses,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }


def predict_model(model, loader):
    model.eval()
    preds = []
    probs = []

    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device)
            logits = model(xb)
            p = torch.softmax(logits, dim=1)
            pred = torch.argmax(p, dim=1)
            preds.extend(pred.cpu().numpy())
            probs.extend(p.cpu().numpy())

    return np.array(preds), np.array(probs)


def summarize_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": acc,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }


Using device: cuda


In [ ]:
# ==============================
# Block 3: 90/10 split-first experiments
# ==============================

# X_train/y_train and X_test/y_test were created in Block 1.
# The split happened on original rows first, then only the training rows were augmented.

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train counts:", pd.Series(y_train).value_counts().sort_index().to_dict())
print("Test counts:", pd.Series(y_test).value_counts().sort_index().to_dict())

optimizers_to_test = ["sgd", "adam", "rmsprop"]
activations_to_test = ["relu", "leakyrelu", "elu"]

results_90_10 = []
trained_runs = {}

for opt_name in optimizers_to_test:
    for act_name in activations_to_test:
        print(f"Training split-first model: optimizer={opt_name}, activation={act_name}")
        run = train_model(
            X_train, y_train, X_test, y_test,
            optimizer_name=opt_name,
            activation=act_name,
            verbose=False
        )
        y_pred = run["y_pred"]
        metrics = summarize_metrics(y_test, y_pred)
        metrics.update({"optimizer": opt_name, "activation": act_name})
        results_90_10.append(metrics)
        trained_runs[(opt_name, act_name)] = run
        print(metrics)

results_90_10_df = pd.DataFrame(results_90_10).sort_values(
    by=["accuracy", "macro_f1"], ascending=False
).reset_index(drop=True)

results_90_10_df


Train shape: (2142, 512) Test shape: (34, 512)
Train counts: {0: 693, 1: 693, 2: 756}
Test counts: {0: 11, 1: 11, 2: 12}
Training split-first model: optimizer=sgd, activation=relu


In [ ]:
# ============================================================
# Paper-style metrics for all saved 90/10 trained models
# Includes NCC, NIC, ACC, PRC, RCL, F1S, INF + correlation matrix
# ============================================================

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Make sure class order matches your notebook labels
# In your notebook, class_names should already exist.
print("Class names:", class_names)

paper_metric_rows = []
confusion_matrices = {}

for (opt_name, act_name), run in trained_runs.items():
    model = run["model"]
    scaler = run["scaler"]

    # Scale test set using the scaler saved for this exact trained model
    X_test_s = scaler.transform(X_test).astype(np.float32)
    X_test_t = torch.tensor(X_test_s).unsqueeze(1)

    test_loader_for_timing = make_loader(
        X_test_t,
        y_test,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # Measure inference time
    start = time.perf_counter()
    y_pred, y_prob = predict_model(model, test_loader_for_timing)
    end = time.perf_counter()

    inf_total_sec = end - start
    inf_per_sample_sec = inf_total_sec / len(y_test)

    cm = confusion_matrix(y_test, y_pred)

    NCC = int(np.trace(cm))
    NIC = int(np.sum(cm) - NCC)

    ACC = accuracy_score(y_test, y_pred)

    # Paper says "overall"; weighted is usually the safest overall metric.
    PRC_weighted = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    RCL_weighted = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    F1S_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    # Macro treats each class equally.
    PRC_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
    RCL_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
    F1S_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

    row = {
        "optimizer": opt_name,
        "activation": act_name,
        "NCC": NCC,
        "NIC": NIC,
        "ACC": ACC,
        "PRC_weighted": PRC_weighted,
        "RCL_weighted": RCL_weighted,
        "F1S_weighted": F1S_weighted,
        "PRC_macro": PRC_macro,
        "RCL_macro": RCL_macro,
        "F1S_macro": F1S_macro,
        "INF_total_sec": inf_total_sec,
        "INF_per_sample_sec": inf_per_sample_sec
    }

    paper_metric_rows.append(row)
    confusion_matrices[(opt_name, act_name)] = cm

paper_metrics_df = pd.DataFrame(paper_metric_rows)

paper_metrics_df = paper_metrics_df.sort_values(
    by=["ACC", "F1S_weighted"],
    ascending=False
).reset_index(drop=True)

print("Paper-style performance indicators for all saved 90/10 models:")
display(paper_metrics_df)

# Save summary table
paper_metrics_df.to_csv("paper_style_metrics_all_90_10_models.csv", index=False)

# ============================================================
# Show confusion matrix for the best model
# ============================================================

best_row = paper_metrics_df.iloc[0]
best_key = (best_row["optimizer"], best_row["activation"])
best_cm = confusion_matrices[best_key]

print("\nBest model:")
print(best_row)

print("\nConfusion matrix for best model:")
best_cm_df = pd.DataFrame(
    best_cm,
    index=[f"Actual_{c}" for c in class_names],
    columns=[f"Predicted_{c}" for c in class_names]
)
display(best_cm_df)

best_cm_df.to_csv("best_model_confusion_matrix.csv")

# ============================================================
# Correlation matrix across model-level numerical metrics
# ============================================================

corr_input = paper_metrics_df.drop(columns=["optimizer", "activation"])

corr_df = corr_input.corr(numeric_only=True)

print("\nCorrelation matrix of numerical model metrics:")
display(corr_df)

corr_df.to_csv("paper_style_metrics_correlation_matrix.csv")

plt.figure(figsize=(10, 8))
plt.imshow(corr_df, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr_df.columns)), corr_df.columns, rotation=45, ha="right")
plt.yticks(range(len(corr_df.index)), corr_df.index)
plt.title("Correlation Matrix of Paper-Style Model Metrics")
plt.tight_layout()
plt.show()

In [ ]:
# Detailed report for the best 90/10 model

best = results_90_10_df.iloc[0]
best_key = (best["optimizer"], best["activation"])
best_run = trained_runs[best_key]

print("Best 90/10 configuration:")
print(best)

print("\nClassification report:")
print(classification_report(y_test, best_run["y_pred"], target_names=class_names, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, best_run["y_pred"]))


In [ ]:
# ==============================
# Block 4: leakage-safe 5-fold cross-validation
# ==============================

# Each fold splits original rows first.
# Then the training fold is augmented.
# The validation fold remains untouched original rows unless AUGMENT_TEST_SET=True.

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = []

for opt_name in optimizers_to_test:
    for act_name in activations_to_test:
        print(f"
Leakage-safe 5-fold CV: optimizer={opt_name}, activation={act_name}")

        for fold, (train_idx, test_idx) in enumerate(skf.split(X_original, y_original), start=1):
            X_tr_orig, X_te_orig = X_original[train_idx], X_original[test_idx]
            y_tr_orig, y_te_orig = y_original[train_idx], y_original[test_idx]
            files_tr_orig = files_original.iloc[train_idx].reset_index(drop=True)
            files_te_orig = files_original.iloc[test_idx].reset_index(drop=True)

            X_tr, y_tr, _, tr_base = augment_split(
                X_tr_orig,
                y_tr_orig,
                files_tr_orig,
                seed=SEED + 1000 + fold,
                include_original=INCLUDE_ORIGINAL_TRAIN,
                split_name=f"cv{fold}_train",
            )

            if AUGMENT_TEST_SET:
                X_te, y_te, _, te_base = augment_split(
                    X_te_orig,
                    y_te_orig,
                    files_te_orig,
                    seed=SEED + 2000 + fold,
                    include_original=False,
                    split_name=f"cv{fold}_test",
                )
            else:
                X_te = X_te_orig.astype(np.float32)
                y_te = y_te_orig.astype(np.int64)
                te_base = files_te_orig

            overlap = set(pd.Series(tr_base).astype(str)).intersection(set(pd.Series(te_base).astype(str)))
            if overlap:
                raise RuntimeError(f"CV leakage detected in fold {fold}: {list(sorted(overlap))[:10]}")

            run = train_model(
                X_tr, y_tr, X_te, y_te,
                optimizer_name=opt_name,
                activation=act_name,
                verbose=False
            )

            metrics = summarize_metrics(y_te, run["y_pred"])
            metrics.update({
                "optimizer": opt_name,
                "activation": act_name,
                "fold": fold,
                "train_n": len(y_tr),
                "test_n": len(y_te),
            })
            cv_results.append(metrics)
            print(f"  fold={fold} train_n={len(y_tr)} test_n={len(y_te)} acc={metrics['accuracy']:.4f} f1={metrics['macro_f1']:.4f}")

cv_results_df = pd.DataFrame(cv_results)

cv_summary_df = (
    cv_results_df
    .groupby(["optimizer", "activation"])
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        macro_precision_mean=("macro_precision", "mean"),
        macro_recall_mean=("macro_recall", "mean"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        train_n_mean=("train_n", "mean"),
        test_n_mean=("test_n", "mean"),
    )
    .reset_index()
    .sort_values(by=["accuracy_mean", "macro_f1_mean"], ascending=False)
)

cv_summary_df


In [ ]:
# Save results for reporting

results_90_10_df.to_csv("cnn_replica_split_then_augment_90_10_results.csv", index=False)
paper_metrics_df.to_csv("paper_style_metrics_split_then_augment_90_10_models.csv", index=False)
cv_results_df.to_csv("cnn_replica_split_then_augment_5fold_raw_results.csv", index=False)
cv_summary_df.to_csv("cnn_replica_split_then_augment_5fold_summary.csv", index=False)

print("Saved:")
print("- cnn_replica_split_then_augment_90_10_results.csv")
print("- paper_style_metrics_split_then_augment_90_10_models.csv")
print("- cnn_replica_split_then_augment_5fold_raw_results.csv")
print("- cnn_replica_split_then_augment_5fold_summary.csv")
